<a href="https://colab.research.google.com/github/xunan007/comp5511-lab/blob/main/Assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1: The Symbolic Architect (GOFAI)
**Course:** COMP5511 - Artificial Intelligence Concepts
**Goal:** Bridge formal logic (SHRDLU) and adversarial search (Gomoku).

In [ ]:
from ipywidgets import Layout, Button, VBox, HBox, Output, Text, HTML
from IPython.display import display, SVG, clear_output
import re
import numpy as np

---
## Part 1: The SHRDLU Micro-world

### Background: What is SHRDLU?
SHRDLU was a groundbreaking natural language understanding program developed by Terry Winograd at MIT in 1968-1970. It allowed users to interact with a virtual "micro-world" containing various colored blocks, pyramids, and boxes using English commands.

In this assignment, we recreate the core logic in two stages:
1.  **Stage 1 (Basic):** A direct command executor (if legal, do it).
2.  **Stage 2 (Advanced):** A planning agent that modifies the world to make an action possible (e.g., clearing the top of a block before using it).

### Task 1.1 - 1.2: World State & Rendering
**Goal:** Represent the physical world symbolically.

We provide the `BlockWorld` class and `render_world` function to visualize this symbolic state.

In [ ]:
class BlockWorld:
    def __init__(self):
        self.blocks = [
            {  "id": "block1", "color": "red", "type": "brick", "x": 50, "y": 150, "width": 30, "height": 60, "on": None, },
            {  "id": "block2", "color": "blue", "type": "square", "x": 150, "y": 150, "width": 40, "height": 40, "on": None, },
            {  "id": "block3", "color": "green", "type": "triangle", "x": 250, "y": 150, "width": 40, "height": 40, "on": None, },
            {  "id": "box1", "color": "gray", "type": "box", "x": 100, "y": 250, "width": 150, "height": 60, "on": None, },
            {  "id": "box2", "color": "gray", "type": "box", "x": 100, "y": 250, "width": 150, "height": 60, "on": None, },
            {  "id": "box3", "color": "gray", "type": "box", "x": 100, "y": 250, "width": 150, "height": 60, "on": None, },
            {  "id": "box4", "color": "gray", "type": "box", "x": 100, "y": 250, "width": 150, "height": 60, "on": None, },
        ]
        self.history = []

    def get_block_by_desc(self, color, shape):
        for b in self.blocks:
            if b["color"] == color and (b["type"] == shape or shape == "object"):
                return b
        return None


def render_world(world):
    svg_template = '<svg width="400" height="400" style="background-color: #f0f0f0; border: 1px solid black;">'
    for b in world.blocks:
        if b["type"] == "triangle":
            points = f"{b['x']},{b['y'] + b['height']} {b['x'] + b['width'] / 2},{b['y']} {b['x'] + b['width']},{b['y'] + b['height']}"
            svg_template += (
                f'<polygon points="{points}" fill="{b["color"]}" stroke="black" />'
            )
        else:
            svg_template += f'<rect x="{b["x"]}" y="{b["y"]}" width="{b["width"]}" height="{b["height"]}" fill="{b["color"]}" stroke="black" />'

        # Add label to distinguish types
        cx = b["x"] + b["width"] / 2
        cy = b["y"] + b["height"] / 2
        svg_template += f'<text x="{cx}" y="{cy}" font-family="monospace" font-size="10" fill="white" stroke="black" stroke-width="0.5" text-anchor="middle" alignment-baseline="middle">{b["type"]}</text>'

    svg_template += "</svg>"
    return SVG(svg_template)

### Task 1.3 - 1.5: Stage 1 - Basic Parser & Action
**Goal:** Translate text into action (Direct Execution).

Understanding a command involves two steps:
1.  **Resolution (`resolve_references`):** Mapping phrases like "the red block" to a concrete object ID. You must implement pronoun resolution ("it").
2.  **Parsing (`simple_parser`):** Use regular expressions to extract the intent (Subject -> Target).
3.  **Action (`move_block`):** This function implements our "intelligence".
    *   **Recursion (Basic):** In Stage 1, if you move a bottom block, the code recursively checks for anything `on` it and moves that too.

In [ ]:
def resolve_references(command, world):
    # TODO: Task 1.3
    # If the command contains the word "it", replace it with the description
    # of the last object mentioned/moved in world.history.
    # Examples:
    #   "Put it on the box" -> "Put red brick on the box"

    # ... your code here ...

    return command


def simple_parser(command, world):
    # TODO: Task 1.4
    # Use regex to parse commands like "Put the [color] [type] on the [color] [type]"
    # pattern = r"..."
    # If match, return the two objects using world.get_block_by_desc()

    # ... your code here ...

    return None, None


def move_block(obj, target, world):
    if obj and target:
        # TODO: Task 1.5 - Recursive "Sticky" Physics
        # 1. Find any object that is currently 'on' this 'obj'.
        # 2. Recursively call move_block on that object to move it to 'obj'
        #    (conceptually keeping them glued).

        # ... your code here ...

        # Execution: Update the state
        obj["on"] = target["id"]

        # TODO: Update obj["x"] and obj["y"]
        # x should be centered relative to the target
        # y should be directly on top of the target (target y - obj height)
        # obj["x"] = ...
        # obj["y"] = ...

        world.history.append(obj)
        return f"OK. Moved {obj['color']} {obj['type']}."
    return "I don't understand that command."

### Testing Stage 1
Let's try out the basic command execution.

In [ ]:
# Simple Test Interface for Stage 1
world_stage1 = BlockWorld()
out_stage1 = Output()
text_input_stage1 = Text(placeholder="Stage 1: e.g. Put the red brick on the gray box")
btn_stage1 = Button(description="Execute (Stage 1)")


def run_stage1(_):
    with out_stage1:
        clear_output(wait=True)
        cmd = resolve_references(text_input_stage1.value, world_stage1)
        o1, o2 = simple_parser(cmd, world_stage1)
        if o1 and o2:
            print(move_block(o1, o2, world_stage1))
        else:
            print("Parser failed to identify objects.")
        display(render_world(world_stage1))


btn_stage1.on_click(run_stage1)
display(
    VBox(
        [
            HTML("<b>Stage 1 Test (Direct Execution)</b>"),
            HBox([text_input_stage1, btn_stage1]),
            out_stage1,
        ]
    )
)
with out_stage1:
    display(render_world(world_stage1))

### Task 1.6: Stage 2 - Advanced Planning (The Logic of "Clearing")
**Goal:** Implement prerequisite checking (Goal-Based Planning).

In the original SHRDLU, if you asked "Put the red block on the green block", and the green block had a pyramid on it, SHRDLU wouldn't just error or glue them together. It would **reason** and **clear** the obstacles first.

You need to implement `clear_top` and `smart_move_block`.

In [ ]:
def get_object_on(obj_id, world):
    """Returns the object sitting on top of obj_id, if any."""
    for b in world.blocks:
        if b["on"] == obj_id:
            return b
    return None


def find_safe_spot(world):
    """Finds an x-location on the floor that is not overlapping with others."""
    max_x = 0
    for b in world.blocks:
        if b["on"] is None and b["y"] > 200:  # Conceptually "on floor"
            right_edge = b["x"] + b["width"]
            if right_edge > max_x:
                max_x = right_edge
    return max_x + 10


def clear_top(obj, world):
    """Recursively moves everything off the top of obj."""
    top_obj = get_object_on(obj["id"], world)

    if top_obj:
        # TODO: Recursive step
        # 1. The object on top must be cleared first.
        # clear_top(..., world)

        # TODO: Move the top_obj to a safe spot
        safe_x = find_safe_spot(world)

        # Update state (conceptually moving it to the floor)
        # top_obj["on"] = None
        # top_obj["x"] = ...
        # top_obj["y"] = ...    # Hint: Floor y is usually around 250-290 depending on height

        world.history.append(top_obj)
        print(f"   [Plan]: Moved covering {top_obj['color']} {top_obj['type']} aside.")


def smart_move_block(obj, target, world):
    if not obj or not target:
        return "I don't understand that command."

    if obj["id"] == target["id"]:
        return "I cannot put a block on itself."

    # 1. Planning: Clear the top of the object we want to move
    # clear_top(obj, world)

    # 2. Planning: Clear the top of the destination
    # clear_top(target, world)

    # 3. Execution: Move the object to the target
    # TODO: Update obj["on"], obj["x"], obj["y"]
    # obj["on"] = ...
    # obj["x"] = ...
    # obj["y"] = ...

    world.history.append(obj)
    return f"OK. Smart-moved {obj['color']} {obj['type']} onto {target['color']} {target['type']}."

### Task 1.7: SHRDLU Interface
**Goal:** The Read-Eval-Print Loop.

In [ ]:
world = BlockWorld()
out = Output()
text_input = Text(placeholder="e.g. Put the red block on the gray box")
btn = Button(description="Action")


def run_shrdlu(_):
    with out:
        clear_output(wait=True)
        cmd = resolve_references(text_input.value, world)
        o1, o2 = simple_parser(cmd, world)
        # Using Stage 2 Logic by default now:
        print(smart_move_block(o1, o2, world))
        display(render_world(world))


btn.on_click(run_shrdlu)
display(text_input, btn, out)
with out:
    display(render_world(world))

---
## Part 2: Gomoku with Robust Minimax

### Background: Gomoku & Adversarial Search
Gomoku (also known as "Five in a Row") is an abstract strategy board game.

#### Two-Stage Approach
1.  **Stage 1 (Basic):** Implement the core Minimax algorithm with basic scoring.
2.  **Stage 2 (Advanced):** Implement **Measures** to optimize depth and speed (Open-ended).

### Task 2.1 - 2.3: Board & Scoring
**Goal:** Define the "Intuition" of the AI.

The `evaluate_board` function allows the AI to judge a board state without seeing the end of the game.

In [ ]:
BOARD_SIZE = 11
AI_GOES_FIRST = False

EMPTY = 0
BLACK = 1  # Black always goes first
WHITE = 2

if AI_GOES_FIRST:
    AI = BLACK
    HUMAN = WHITE
else:
    HUMAN = BLACK
    AI = WHITE


def check_win(board, player):
    for r in range(BOARD_SIZE):
        for c in range(BOARD_SIZE):
            if board[r][c] != player:
                continue
            for dr, dc in [(0, 1), (1, 0), (1, 1), (1, -1)]:
                if all(
                    0 <= r + i * dr < BOARD_SIZE
                    and 0 <= c + i * dc < BOARD_SIZE
                    and board[r + i * dr][c + i * dc] == player
                    for i in range(5)
                ):
                    return True
    return False


def get_line_score(line):
    # Count occurrences of AI and Human in this segment of 5 spots
    ai_cnt = line.count(AI)
    human_cnt = line.count(HUMAN)

    # Mixed line (dead end for both)
    if ai_cnt > 0 and human_cnt > 0:
        return 0

    # TODO: Task 2.2 - Assign scores
    # Return high positive values if AI is winning/strong.
    # Return high NEGATIVE values if Human is winning/strong.
    # Special attention: If Human has 4 in a row, you MUST return a huge negative score
    # to force validity in Minimax.
    if ai_cnt == 5:
        return 1000000
    # ... complete the logic ...

    return 0


def evaluate_board(board):
    score = 0
    for r in range(BOARD_SIZE):
        for c in range(BOARD_SIZE):
            for dr, dc in [(0, 1), (1, 0), (1, 1), (1, -1)]:
                if 0 <= r + 4 * dr < BOARD_SIZE and 0 <= c + 4 * dc < BOARD_SIZE:
                    line = tuple(board[r + i * dr][c + i * dc] for i in range(5))
                    score += get_line_score(line)
                    del line
    return score

### Task 2.4 - 2.6: Stage 1 - Basic Alpha-Beta Search
**Goal:** Implement the decision tree.

**Minimax** is a recursive algorithm. **Alpha-Beta Pruning** is an optimization to stop searching bad branches.

In [ ]:
def minimax(board, depth, alpha, beta, is_max):
    # TODO: Task 2.4
    # 1. Base Case: If depth == 0 or game over, return evaluate_board()

    found_move = False

    if is_max:  # AI Turn
        val = -float("inf")
        # TODO: Iterate over all cells
        # If empty:
        #   Place AI piece
        #   val = max(val, recursive_call)
        #   Remove piece
        #   Update alpha
        #   If beta <= alpha: break
        pass

    else:  # Human Turn (Minimizing)
        val = float("inf")
        # TODO: Iterate over all cells
        # If empty:
        #   Place HUMAN piece
        #   val = min(val, recursive_call)
        #   Remove piece
        #   Update beta
        #   If beta <= alpha: break
        pass

    # Handle scenario where no moves are possible
    return val if found_move else 0


def get_best_move(board, depth=2):
    best_val = -float("inf")
    move = (-1, -1)
    # Simple optimization: Only search near existing pieces could be added here
    # For now, search all empty spots (or implement the neighbor check)
    for r in range(BOARD_SIZE):
        for c in range(BOARD_SIZE):
            if board[r][c] == EMPTY:
                board[r][c] = AI
                val = minimax(board, depth, -float("inf"), float("inf"), False)
                board[r][c] = EMPTY
                if val > best_val:
                    best_val = val
                    move = (r, c)
    return move

### Testing Stage 1
Run below code. An interface is provided to allow you to play against AI.

In [ ]:
# @title
# --- Game Interface Setup (Provided) ---
GAME_STYLE = """<style>
    .grid-square { font-size: 16px !important; }
    .black-piece { color: black !important; }
    .white-piece { color: white !important; }
</style>"""


class GomokuGame:
    def __init__(self, ai_function, depth=2, title="Gomoku"):
        self.ai_function = ai_function
        self.depth = depth
        self.board = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=int).tolist()
        self.finished = False
        self.out = Output()
        self.title = title

        # Create Grid
        grid_rows = []
        self.buttons = []
        for r in range(BOARD_SIZE):
            row_btns = []
            for c in range(BOARD_SIZE):
                btn = Button(
                    layout=Layout(
                        width="30px",
                        height="30px",
                        margin="0px",
                        padding="0px",
                        border="1px solid #555",
                    ),
                    tooltip=f"{r},{c}",
                )
                btn.style.button_color = "#DEB887"
                btn.add_class("grid-square")
                btn.on_click(self.on_click)
                row_btns.append(btn)
            self.buttons.append(row_btns)
            grid_rows.append(HBox(row_btns))

        self.board_widget = VBox(grid_rows)
        # Main Display
        self.widget = VBox(
            [
                HTML(value=GAME_STYLE),
                HTML(f"<b>{title}</b>"),
                self.out,
                self.board_widget,
            ]
        )
        self.log("New Game. Black (You) vs White (AI). Click to place stone.")

    def log(self, msg):
        with self.out:
            clear_output(wait=True)
            print(msg)

    def update_visual(self, r, c, player):
        btn = self.buttons[r][c]
        btn.description = "⬤"
        btn.remove_class("black-piece")
        btn.remove_class("white-piece")

        if player == BLACK:
            btn.add_class("black-piece")
            btn.style.text_color = "black"
        else:
            btn.add_class("white-piece")
            btn.style.text_color = "white"

    def on_click(self, btn):
        if self.finished:
            return
        r, c = map(int, btn.tooltip.split(","))
        if self.board[r][c] != EMPTY:
            return

        # Human Move
        self.board[r][c] = HUMAN
        self.update_visual(r, c, HUMAN)
        if check_win(self.board, HUMAN):
            self.log("You Win!")
            self.finished = True
            return

        self.log(f"AI Thinking (Depth {self.depth})...")

        # AI Move
        move = self.ai_function(self.board, depth=self.depth)

        if move != (-1, -1):
            ar, ac = move
            self.board[ar][ac] = AI
            self.update_visual(ar, ac, AI)
            if check_win(self.board, AI):
                self.log("AI Wins!")
                self.finished = True
            else:
                self.log("Your turn.")
        else:
            self.log("Draw / No moves.")


# Launch Stage 1 Game
game_s1 = GomokuGame(get_best_move, depth=1, title="Stage 1: Basic AI (Click to Play)")
display(game_s1.widget)

### Task 2.7: Stage 2 - Optimization
**Goal:** Prune the tree faster and evaluate board states smarter.

**Move Ordering:** Alpha-Beta pruning works best when we find a good move *early*.
**Heuristics:** Center positions might be better than edges.

This stage is open-ended. You should design your own `get_best_move_advanced` function.
It might involve creating a helper to sort moves (Move Ordering) or updating the evaluation function.

In [ ]:
# TODO: Task 2.7 - Implement Advanced Minimax with Move Ordering & Heuristics
# Goals:
# 1. Improve Evaluation: Add heuristics (e.g. center control is valuable).
# 2. Improve Search: Use "Move Ordering" to check promising moves first.


def evaluate_board_advanced(board):
    """
    Improved evaluation function.
    """
    # Start with the basic line score
    score = evaluate_board(board)

    # TODO: Add Heuristics (Pro Tip: Positional Heuristic)
    # Strategy: Stones in the center of the board are generally more valuable.
    # 1. Define a weights matrix or calculate distance from center.
    # 2. Add bonus points to 'score' for AI stones in good spots.
    # 3. Subtract points for Human stones in good spots.

    return score


def get_promising_moves(board, player):
    """
    Optional Helper: Returns a list of (r, c) tuples worth checking.
    """
    candidates = []
    # TODO: Optimize Search Space
    # Instead of checking every empty spot, only check spots that have
    # an existing neighbor within radius 1 or 2. This is "spatial locality".

    # TODO: (Optional) Sort candidates by a quick heuristic (e.g. immediate value)
    # candidates.sort(...)

    return candidates


def minimax_advanced(board, depth, alpha, beta, is_max):
    # TODO: Duplicate your minimax logic here, but:
    # 1. Use evaluate_board_advanced() at the leaf nodes.
    # 2. Use get_promising_moves() in the loop to look at good moves first.
    return 0


def get_best_move_advanced(board, depth=2):
    # 1. Get Promising Moves
    # moves = get_promising_moves(board, AI)
    # if not moves: return (BOARD_SIZE // 2, BOARD_SIZE // 2)

    best_val = -float("inf")
    best_move = (-1, -1)

    alpha = -float("inf")
    beta = float("inf")

    # 2. Iterate through ordered moves
    # for r, c in moves:
    #     board[r][c] = AI
    #     val = minimax_advanced(board, depth - 1, alpha, beta, False)
    #     board[r][c] = EMPTY
    #
    #     if val > best_val:
    #         best_val = val
    #         best_move = (r, c)
    #     alpha = max(alpha, best_val)

    return best_move

### Testing the Final Version

In [ ]:
game_s2 = GomokuGame(
    get_best_move_advanced, depth=2, title="Stage 2: Advanced AI (Click to Play)"
)
display(game_s2.widget)